# 02 - Vehicle Location Extraction 


In [1]:
import os
import glob
import xml.etree.ElementTree as ET

import pandas as pd
from pyspark.sql import SparkSession


In [2]:
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("VehicleLocation")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.local.ip", "127.0.0.1")
    .config("spark.driver.memory", "4g")
    .config("spark.python.worker.reuse", "true")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


In [3]:
project_path = r"D:\BigDataCoursework"

vehicle_path = os.path.join(project_path, "Data", "raw", "vehicle_location")

vehicle_files = glob.glob(os.path.join(vehicle_path, "**", "*.xml"), recursive=True)
print("Vehicle XML Files Found:", len(vehicle_files))


Vehicle XML Files Found: 1


In [4]:
SIRI_NS = {"siri": "http://www.siri.org.uk/siri"}


In [5]:
vehicle_records = []

for file in vehicle_files:
    tree = ET.parse(file)
    root = tree.getroot()

    for activity in root.findall(".//siri:VehicleActivity", SIRI_NS):
        journey = activity.find("siri:MonitoredVehicleJourney", SIRI_NS)
        if journey is None:
            continue

        recorded_at = activity.findtext("siri:RecordedAtTime", default=None, namespaces=SIRI_NS)
        location = journey.find("siri:VehicleLocation", SIRI_NS)

        vehicle_records.append((
            journey.findtext("siri:VehicleJourneyRef", default="Unknown", namespaces=SIRI_NS),
            journey.findtext("siri:OperatorRef", default="Unknown", namespaces=SIRI_NS),
            journey.findtext("siri:LineRef", default="Unknown", namespaces=SIRI_NS),
            journey.findtext("siri:PublishedLineName", default="Unknown", namespaces=SIRI_NS),
            journey.findtext("siri:DirectionRef", default="Unknown", namespaces=SIRI_NS),
            journey.findtext("siri:OriginName", default="Unknown", namespaces=SIRI_NS),
            journey.findtext("siri:DestinationName", default="Unknown", namespaces=SIRI_NS),
            location.findtext("siri:Latitude", default=None, namespaces=SIRI_NS) if location is not None else None,
            location.findtext("siri:Longitude", default=None, namespaces=SIRI_NS) if location is not None else None,
            recorded_at,
        ))

print("Total Vehicle Activity Records:", len(vehicle_records))


Total Vehicle Activity Records: 27987


In [6]:
vehicle_df = pd.DataFrame(
    vehicle_records,
    columns=["Journey_ID", "Operator_ID", "Line_Ref", "Line_Name", "Direction",
             "Origin", "Destination", "Latitude", "Longitude", "Recorded_At"],
)

vehicle_df["Latitude"] = pd.to_numeric(vehicle_df["Latitude"], errors="coerce")
vehicle_df["Longitude"] = pd.to_numeric(vehicle_df["Longitude"], errors="coerce")
vehicle_df["Recorded_At"] = pd.to_datetime(vehicle_df["Recorded_At"], errors="coerce")

vehicle_df = vehicle_df.dropna(subset=["Latitude", "Longitude"])
print(vehicle_df.shape)
vehicle_df.head(10)


(27987, 10)


,Journey_ID,Operator_ID,Line_Ref,Line_Name,Direction,Origin,Destination,Latitude,Longitude,Recorded_At
0,1314,A2BR,10,10,outbound,Market Street,Market Street,52.394055,0.243299,2026-07-16 12:57:22+00:00
1,1414,A2BR,17,17,outbound,Congregational Church,Bus Station,52.065041,-0.113580,2026-07-16 13:23:18+00:00
2,Unknown,A2BV,811,811,outbound,Knutsford_Road,Tesco,53.168910,-2.974826,2026-07-16 07:46:36+00:00
3,Unknown,A2BV,106,106,outbound,Monk_Road,Dominick_House,53.420296,-3.035593,2026-07-16 07:23:48+00:00
4,Unknown,A2BV,175,175,outbound,Heswall_Shore,Heswall_Shore,53.352751,-3.117403,2026-07-16 11:59:42+00:00
5,Unknown,A2BV,175,175,outbound,Heswall_Shore,Heswall_Shore,53.326888,-3.113863,2026-07-16 13:23:10+00:00
6,Unknown,A2BV,1B,1B,outbound,Silverburn_Avenue,Silverburn_Avenue,53.402375,-3.112111,2026-07-16 06:33:28+00:00
7,Unknown,A2BV,91,91,outbound,Birkenhead_Bus_Station,Birkenhead_Bus_Station,53.388253,-3.031908,2026-07-16 13:23:18+00:00
8,Unknown,A2BV,81,81,outbound,West_Kirby_Station,West_Kirby_Station,53.373068,-3.182128,2026-07-16 13:23:27+00:00
9,Unknown,A2BV,1,1,outbound,Garden_Lane,Silverburn_Avenue,53.414285,-3.097011,2026-07-16 07:46:13+00:00


In [7]:
processed_path = os.path.join(project_path, "Data", "processed")

csv_path = os.path.join(processed_path, "vehicle_locations.csv")
vehicle_df.to_csv(csv_path, index=False)
print("Vehicle locations CSV saved:", csv_path)

vehicle_sdf = spark.createDataFrame(vehicle_df.astype(str))
vehicle_sdf = vehicle_sdf.repartition(max(4, spark.sparkContext.defaultParallelism))
print("Vehicle Parquet partitions:", vehicle_sdf.rdd.getNumPartitions())

vehicle_sdf.write.mode("overwrite").parquet(os.path.join(processed_path, "vehicle_locations.parquet"))
print("Vehicle locations Parquet saved.")


Vehicle locations CSV saved: D:\BigDataCoursework\Data\processed\vehicle_locations.csv
Vehicle Parquet partitions: 16
Vehicle locations Parquet saved.


In [8]:
import folium
from folium.plugins import MarkerCluster

uk_map = folium.Map(location=[54.5, -2.5], zoom_start=6, tiles="CartoDB positron")
marker_cluster = MarkerCluster().add_to(uk_map)

for _, row in vehicle_df.head(5000).iterrows():
    folium.CircleMarker(
        location=[row["Latitude"], row["Longitude"]],
        radius=3,
        popup=f"""<b>Line:</b> {row['Line_Name']}<br>
        <b>Operator:</b> {row['Operator_ID']}<br>
        <b>Origin:</b> {row['Origin']}<br>
        <b>Destination:</b> {row['Destination']}""",
        color="blue", fill=True, fill_opacity=0.7,
    ).add_to(marker_cluster)

map_path = os.path.join(processed_path, "vehicle_map.html")
uk_map.save(map_path)
print("Map saved:", map_path)


Map saved: D:\BigDataCoursework\Data\processed\vehicle_map.html


In [9]:
spark.stop()
print("Spark stopped successfully.")


Spark stopped successfully.
